<!--
    Copyright (C) 2026 The AiQuokkaWannabe Authors <https://github.com/ai-quokka-wannabe/iso-lab>

    This program is free software: you can redistribute it and/or modify it under the terms of
    the GNU General Public License as published by the Free Software Foundation, either version
    3 of the License, or (at your option) any later version.

    This program is distributed in the hope that it will be useful, but WITHOUT ANY WARRANTY;
    without even the implied warranty of MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.
    See the GNU General Public License for more details.

    You should have received a copy of the GNU General Public License along with this program.
    If not, see <https://www.gnu.org/licenses/>.
-->

# The Midbody Speed, Judged by the States It Gives

The check written down in the journal of 2026-10-15, *The Midbody Speed, Judged by the States It Gives*, before it
ran: on the seventeen records of the speed check, the lab's locomotion states (the Rust binary `locomotion-states`,
from its eight-arc speed - both versions - and eight-arc length) against the published `motion_modes`. Per record and
state (forward, backward, paused), the share of frames in that state, and the difference from the published share. A
speed passes if, for each state, the median over records of the absolute difference is at most a quarter of the sample
standard deviation (n - 1) of the published share across the records, and no record's absolute difference exceeds
half of it. The frame-by-frame agreement, undefined as a class of its own, is reported, not tested.

Run from the repository's root with the block-1 files in `data/raw/owmd-n2/` (fetched by `fetch-block`, which names
a file only once its MD5 matches) and the Rust toolchain installed; the notebook builds and calls the binary.


In [1]:
import os, subprocess
import numpy as np
import h5py

os.chdir(os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd())
DIRECTORY, SUFFIX = "data/raw/owmd-n2", "_features.hdf5"
# The seventeen records of the speed check of 2026-10-14, pinned so that a re-run checks the same set.
records = ["1006562", "1007123", "1007405", "1008698", "1011246", "1011901", "1013148", "1014831", "1020507",
           "1021835", "1024910", "1025254", "1025397", "1029332", "1030984", "1031252", "1032937"]
missing = [record for record in records if not os.path.exists(f"{DIRECTORY}/{record}{SUFFIX}")]
assert not missing, f"fetch these first: {missing}"
subprocess.run(["cargo", "build", "--quiet", "--release", "--bin", "locomotion-states"], check=True)
print(len(records), "records checked")
print("h5py", h5py.__version__, "numpy", np.__version__)

17 records checked
h5py 3.16.0 numpy 2.5.4


In [2]:
STATES = {"forward": 1, "backward": -1, "paused": 0}

def lab_states(record, published_choices):
    flags = ["--published-choices"] if published_choices else []
    out = subprocess.run(["target/release/locomotion-states", *flags, record], check=True, capture_output=True,
                         text=True).stdout
    return np.array([float(line) for line in out.splitlines()])

published = {}
for record in records:
    with h5py.File(f"{DIRECTORY}/{record}{SUFFIX}", "r") as file:
        published[record] = file["/features_timeseries"]["motion_modes"].astype(float)

def share(modes, value):
    return np.mean(modes == value)

theirs = {name: np.array([share(published[r], v) for r in records]) for name, v in STATES.items()}
spread = {name: np.std(theirs[name], ddof=1) for name in STATES}
print("published share per state, mean and sample SD over records:")
for name in STATES:
    print(f"  {name:8s} mean {theirs[name].mean():.4f}  SD {spread[name]:.4f}")
print()

for choices, title in ((False, "the measure of 2026-10-14"), (True, "the published choices")):
    ours = {r: lab_states(r, choices) for r in records}
    for r in records:
        assert len(ours[r]) == len(published[r]), (r, len(ours[r]), len(published[r]))
    print(title)
    passes = True
    for name, v in STATES.items():
        diff = np.array([share(ours[r], v) for r in records]) - theirs[name]
        median, worst = np.median(np.abs(diff)), np.max(np.abs(diff))
        ok = median <= spread[name] / 4 and worst <= spread[name] / 2
        passes &= ok
        print(f"  {name:8s} median |diff| {median:.4f} (bar {spread[name] / 4:.4f})  max |diff| {worst:.4f} "
              f"(bar {spread[name] / 2:.4f}, record {records[int(np.argmax(np.abs(diff)))]})  passes {ok}")
    code = lambda m: np.where(np.isnan(m), 9, m).astype(int)
    agreement = [np.mean(code(ours[r]) == code(published[r])) for r in records]
    print(f"  frame agreement (undefined as a class): median {np.median(agreement):.4f}, "
          f"range {min(agreement):.4f} to {max(agreement):.4f}")
    print("  passes:", passes)
    print()

published share per state, mean and sample SD over records:
  forward  mean 0.3874  SD 0.2393
  backward mean 0.1034  SD 0.0373
  paused   mean 0.3783  SD 0.2045



the measure of 2026-10-14
  forward  median |diff| 0.0544 (bar 0.0598)  max |diff| 0.1829 (bar 0.1196, record 1031252)  passes False
  backward median |diff| 0.0176 (bar 0.0093)  max |diff| 0.0376 (bar 0.0186, record 1029332)  passes False
  paused   median |diff| 0.0604 (bar 0.0511)  max |diff| 0.1056 (bar 0.1022, record 1013148)  passes False
  frame agreement (undefined as a class): median 0.8181, range 0.7360 to 0.9199
  passes: False



the published choices
  forward  median |diff| 0.0093 (bar 0.0598)  max |diff| 0.0228 (bar 0.1196, record 1011901)  passes True
  backward median |diff| 0.0028 (bar 0.0093)  max |diff| 0.0111 (bar 0.0186, record 1007405)  passes True
  paused   median |diff| 0.0473 (bar 0.0511)  max |diff| 0.0842 (bar 0.1022, record 1011901)  passes True
  frame agreement (undefined as a class): median 0.8970, range 0.8440 to 0.9739
  passes: True

